# Ch.16 — TensorBoard  ·  *PyTorch Companion*

> This is the PyTorch companion to [notebook.ipynb](notebook.ipynb). TensorBoard is
> framework-agnostic — the event-file format is shared — so the **same
> `tensorboard --logdir logs/` command visualises both TF and PyTorch runs side-by-side**.
>
> The PyTorch API is `torch.utils.tensorboard.SummaryWriter`, and where Keras uses a
> `TensorBoard` callback with `histogram_freq` / `update_freq` flags, PyTorch makes us call
> `writer.add_scalar` / `writer.add_histogram` / `writer.add_embedding` explicitly inside the
> training loop. That explicitness is the main pedagogical difference covered below.

## Core Idea

TensorBoard reads event files written by TF/PyTorch and renders:

| Panel | Logging trigger | Primary diagnostic |
|---|---|---|
| Scalars | `update_freq='epoch'` | Overfitting gap, convergence rate |
| Histograms | `histogram_freq=1` | Vanishing/exploding gradients |
| Projector | manual `np.savetxt` | Are learned embeddings meaningful? |
| Graphs | `write_graph=True` | Architecture verification |

```
tensorboard --logdir logs/
```
Then open `http://localhost:6006`

## Running Example

Dataset: **California Housing** (sklearn).  
Model: 3-layer dense network (same as Ch.5).  
Goal: instrument training with TensorBoard and read each diagnostic panel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"TensorFlow version: {tf.__version__}")

# ── Data ──────────────────────────────────────────────────────────────────────
housing    = fetch_california_housing()
X, y_raw   = housing.data, housing.target
y          = y_raw.reshape(-1, 1)

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

X_tr, X_te, y_tr, y_te = train_test_split(X_sc, y,   test_size=0.2, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape[0]:,}  Val: {X_va.shape[0]:,}  Test: {X_te.shape[0]:,}")

### PyTorch equivalent — imports + `SummaryWriter`

| Keras | PyTorch |
|---|---|
| `keras.callbacks.TensorBoard(log_dir=...)` | `SummaryWriter(log_dir=...)` + manual `add_scalar` calls |
| `histogram_freq=1` | call `writer.add_histogram(name, tensor, step)` yourself each epoch |
| `write_graph=True` | `writer.add_graph(model, example_input)` (call once) |
| scalars logged automatically | call `writer.add_scalar('loss/train', v, step)` per epoch |

Data vars (`X_tr`, `X_va`, `X_te`, `y_tr`, `y_va`, `y_te`, `scaler`) are already loaded by the
TF cell above — we just wrap them in tensors.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} available (device={device})")

# Wrap the sklearn arrays once — reused throughout the PyTorch cells below
Xtr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
ytr_t = torch.tensor(y_tr, dtype=torch.float32, device=device)
Xva_t = torch.tensor(X_va, dtype=torch.float32, device=device)
yva_t = torch.tensor(y_va, dtype=torch.float32, device=device)
Xte_t = torch.tensor(X_te, dtype=torch.float32, device=device)
yte_t = torch.tensor(y_te, dtype=torch.float32, device=device)

loader_tb = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=256, shuffle=True)

## Build the Model (Ch.5 Architecture)

In [ ]:
def build_model(learning_rate=1e-3, use_batchnorm=False):
    layers = [
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(64, activation='relu', name='hidden_1',
                           kernel_initializer='he_normal'),
    ]
    if use_batchnorm:
        layers.append(keras.layers.BatchNormalization())
    layers += [
        keras.layers.Dense(32, activation='relu', name='hidden_2',
                           kernel_initializer='he_normal'),
        keras.layers.Dense(16, activation='relu', name='hidden_3',
                           kernel_initializer='he_normal'),
        keras.layers.Dense(1,                     name='output'),
    ]
    model = keras.Sequential(layers)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model

model = build_model()
model.summary()

### PyTorch equivalent — model (Ch.5 architecture)

In [ ]:
class HousingNet(nn.Module):
    """Same 64→32→16→1 architecture as the Keras model above.
    Named submodules so `writer.add_histogram('hidden_1/weight', ...)` is readable.
    """

    def __init__(self, in_features, use_batchnorm=False):
        super().__init__()
        self.hidden_1 = nn.Linear(in_features, 64)
        self.bn       = nn.BatchNorm1d(64) if use_batchnorm else nn.Identity()
        self.hidden_2 = nn.Linear(64, 32)
        self.hidden_3 = nn.Linear(32, 16)
        self.out      = nn.Linear(16, 1)

        # He-normal init, matching Keras `kernel_initializer='he_normal'`
        for m in (self.hidden_1, self.hidden_2, self.hidden_3):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x, return_embedding=False):
        h = F.relu(self.hidden_1(x))
        h = self.bn(h)
        h = F.relu(self.hidden_2(h))
        h3 = F.relu(self.hidden_3(h))
        y  = self.out(h3)
        return (y, h3) if return_embedding else y


torch_model = HousingNet(in_features=X_tr.shape[1]).to(device)
print(torch_model)
print(f"\nTotal parameters: {sum(p.numel() for p in torch_model.parameters()):,}")

## TensorBoard Callback: Scalars + Histograms

The `TensorBoard` callback writes event files to `log_dir`.  
After training, open a terminal and run:  
```
tensorboard --logdir logs/
```
then navigate to `http://localhost:6006`.

In [ ]:
run_id  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = os.path.join('logs', f'ch16_{run_id}')

tb_callback = keras.callbacks.TensorBoard(
    log_dir        = log_dir,
    histogram_freq = 1,          # weight + bias histograms per epoch
    write_graph    = True,       # log computational graph (once)
    update_freq    = 'epoch',    # scalars per epoch — NOT per batch
    profile_batch  = 0,          # disable profiling (set to 2 to profile batch 2)
)

history = model.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = 60,
    batch_size      = 256,
    callbacks       = [tb_callback],
    verbose         = 0
)

print(f"Training complete. Logs: {log_dir}")
print(f"Run:  tensorboard --logdir logs/")

### PyTorch equivalent — training with `SummaryWriter`

Every Keras `TensorBoard` callback feature is a single line in PyTorch, just made explicit.

In [ ]:
run_id_t  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_torch'
log_dir_t = os.path.join('logs', f'ch16_{run_id_t}')
writer    = SummaryWriter(log_dir=log_dir_t)

# Graph panel — log once with an example input
writer.add_graph(torch_model, Xtr_t[:1])

loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)

history_t = {"loss": [], "val_loss": [], "mae": [], "val_mae": []}
for epoch in range(60):
    torch_model.train()
    for xb, yb in loader_tb:
        optimizer.zero_grad()
        loss_fn(torch_model(xb), yb).backward()
        optimizer.step()

    torch_model.eval()
    with torch.no_grad():
        tr_pred = torch_model(Xtr_t); va_pred = torch_model(Xva_t)
        tr_mse  = loss_fn(tr_pred, ytr_t).item()
        va_mse  = loss_fn(va_pred, yva_t).item()
        tr_mae  = (tr_pred - ytr_t).abs().mean().item()
        va_mae  = (va_pred - yva_t).abs().mean().item()

    history_t["loss"].append(tr_mse);     history_t["val_loss"].append(va_mse)
    history_t["mae"].append(tr_mae);      history_t["val_mae"].append(va_mae)

    # Scalars (update_freq='epoch' equivalent)
    writer.add_scalar('loss/train', tr_mse, epoch)
    writer.add_scalar('loss/val',   va_mse, epoch)
    writer.add_scalar('mae/train',  tr_mae, epoch)
    writer.add_scalar('mae/val',    va_mae, epoch)

    # Histograms (histogram_freq=1 equivalent)
    for name, param in torch_model.named_parameters():
        writer.add_histogram(name, param, epoch)
        if param.grad is not None:
            writer.add_histogram(f'{name}.grad', param.grad, epoch)

writer.flush()
print(f"PyTorch training complete. Logs: {log_dir_t}")
print(f"Run:  tensorboard --logdir logs/   (both TF and PyTorch runs will appear)")

## Reading the Scalars Panel (in Notebook)

We can also plot the logged metrics directly from the `history` object.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history.history['loss']) + 1)

ax1.plot(epochs, history.history['loss'],     label='train MSE', lw=1.5)
ax1.plot(epochs, history.history['val_loss'], label='val MSE',   lw=1.5, linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title('Loss (MSE) curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history.history['mae'],     label='train MAE', lw=1.5)
ax2.plot(epochs, history.history['val_mae'], label='val MAE',   lw=1.5, linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE')
ax2.set_title('MAE curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Scalars panel — same curves visible in TensorBoard', y=1.02)
plt.tight_layout(); plt.show()

final_val_rmse = history.history['val_loss'][-1] ** 0.5
print(f"Final validation RMSE: {final_val_rmse:.4f}")

### PyTorch equivalent — plot from `history_t`

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs_t = range(1, len(history_t['loss']) + 1)

ax1.plot(epochs_t, history_t['loss'],     label='train MSE', lw=1.5)
ax1.plot(epochs_t, history_t['val_loss'], label='val MSE',   lw=1.5, linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title('PyTorch — Loss (MSE) curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_t, history_t['mae'],     label='train MAE', lw=1.5)
ax2.plot(epochs_t, history_t['val_mae'], label='val MAE',   lw=1.5, linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE')
ax2.set_title('PyTorch — MAE curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('PyTorch Scalars panel — same curves visible in TensorBoard', y=1.02)
plt.tight_layout(); plt.show()
print(f"PyTorch final validation RMSE: {history_t['val_loss'][-1] ** 0.5:.4f}")

## Custom Scalars: Learning Rate Logger

The TensorBoard callback doesn't log LR automatically. Write a custom callback using `tf.summary`.

In [ ]:
class LRSchedulerLogger(keras.callbacks.Callback):
    """Logs current learning rate to TensorBoard Scalars."""
    def __init__(self, log_dir):
        super().__init__()
        self.writer = tf.summary.create_file_writer(os.path.join(log_dir, 'custom'))

    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        with self.writer.as_default():
            tf.summary.scalar('learning_rate', lr, step=epoch)
        self.writer.flush()


# ── Model with cosine-decay LR schedule ──────────────────────────────────────
run_id2  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_cosine'
log_dir2 = os.path.join('logs', f'ch16_{run_id2}')

cosine_lr  = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=60 * (len(X_tr) // 256)
)
model2 = build_model()
model2.compile(optimizer=keras.optimizers.Adam(cosine_lr), loss='mse', metrics=['mae'])

hist2 = model2.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = 60,
    batch_size      = 256,
    callbacks       = [
        keras.callbacks.TensorBoard(log_dir=log_dir2, histogram_freq=5, update_freq='epoch'),
        LRSchedulerLogger(log_dir2),
    ],
    verbose = 0
)
print(f"Cosine LR run written to: {log_dir2}")

### PyTorch equivalent — cosine LR + scalar logging

No custom callback needed — `torch.optim.lr_scheduler.CosineAnnealingLR` plus one explicit
`add_scalar('learning_rate', ...)` call per epoch does the same job as Keras's
`LRSchedulerLogger`.

In [ ]:
run_id2_t  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '_cosine_torch'
log_dir2_t = os.path.join('logs', f'ch16_{run_id2_t}')
writer2    = SummaryWriter(log_dir=log_dir2_t)

torch.manual_seed(42)
model2_t  = HousingNet(in_features=X_tr.shape[1]).to(device)
optim2    = torch.optim.Adam(model2_t.parameters(), lr=1e-3)
# CosineAnnealingLR decays lr from initial to eta_min over T_max steps.
# Keras decayed over 60 * (len(X_tr) // 256) optimizer steps; we step the scheduler per batch.
n_steps   = 60 * (len(X_tr) // 256)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim2, T_max=n_steps, eta_min=0.0)

hist2_t = {"val_loss": []}
for epoch in range(60):
    model2_t.train()
    for xb, yb in loader_tb:
        optim2.zero_grad()
        loss_fn(model2_t(xb), yb).backward()
        optim2.step()
        scheduler.step()           # per-batch cosine decay (matches Keras CosineDecay)

    model2_t.eval()
    with torch.no_grad():
        va = loss_fn(model2_t(Xva_t), yva_t).item()
    hist2_t["val_loss"].append(va)

    writer2.add_scalar('loss/val',       va,                 epoch)
    writer2.add_scalar('learning_rate',  scheduler.get_last_lr()[0], epoch)

writer2.flush()
print(f"PyTorch cosine-LR run written to: {log_dir2_t}")

## Comparing Two Runs in TensorBoard

When `--logdir logs/` is used, TensorBoard shows all child runs together. Compare Adam with constant LR vs Adam with cosine decay.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_range = range(1, 61)
ax.plot(epochs_range, history.history['val_loss'],  label='Adam constant LR',  lw=1.5)
ax.plot(epochs_range, hist2.history['val_loss'],    label='Adam cosine LR',    lw=1.5, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE')
ax.set_title('Run comparison — same chart shown in TensorBoard Scalars')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("In TensorBoard: open Scalars tab, select both runs — curves overlay automatically.")

### PyTorch equivalent — run comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, 61), history_t['val_loss'], label='PyTorch constant LR', lw=1.5)
ax.plot(range(1, 61), hist2_t['val_loss'],   label='PyTorch cosine LR',   lw=1.5, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE')
ax.set_title('PyTorch — Run comparison (same data shown in TensorBoard Scalars)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Open TensorBoard → Scalars → tick both PyTorch runs to overlay in-browser.")

## Projector: Log Intermediate Embeddings

Extract the hidden_3 layer's 16-dimensional activations for 500 test samples. Write them as TSV files for the TensorBoard Projector tab.

In [ ]:
# ── Extract layer 3 activations ───────────────────────────────────────────────
embedding_model = keras.Model(
    inputs  = model.input,
    outputs = model.get_layer('hidden_3').output
)
n_emb      = 500
embeddings = embedding_model.predict(X_te[:n_emb], verbose=0)   # (500, 16)
print(f"Embedding tensor shape: {embeddings.shape}")

# ── Write TSV files ───────────────────────────────────────────────────────────
proj_dir = os.path.join(log_dir, 'projector')
os.makedirs(proj_dir, exist_ok=True)

np.savetxt(os.path.join(proj_dir, 'feature_vecs.tsv'),
           embeddings, delimiter='\t', fmt='%.6f')

# Metadata: house value quantile (low / mid / high) for colouring
y_te_flat = y_te[:n_emb, 0]
q33, q67  = np.percentile(y_te_flat, [33, 67])
with open(os.path.join(proj_dir, 'metadata.tsv'), 'w') as f:
    f.write('MedHouseVal\tValueBand\n')
    for v in y_te_flat:
        band = 'Low' if v < q33 else ('High' if v > q67 else 'Mid')
        f.write(f'{v:.3f}\t{band}\n')

# Write projector_config.pbtxt
config_txt = f"""embeddings {{
  tensor_path: "feature_vecs.tsv"
  metadata_path: "metadata.tsv"
}}"""
with open(os.path.join(proj_dir, 'projector_config.pbtxt'), 'w') as f:
    f.write(config_txt)

print(f"Projector files written to: {proj_dir}")
print("In TensorBoard: open Projector tab → load config from this directory.")

### PyTorch equivalent — Projector via `writer.add_embedding`

PyTorch skips the TSV-file dance: `SummaryWriter.add_embedding(mat, metadata, ...)` writes all
the Projector artifacts in one call.

In [ ]:
n_emb = 500
torch_model.eval()
with torch.no_grad():
    _, embeddings_t = torch_model(Xte_t[:n_emb], return_embedding=True)
embeddings_t = embeddings_t.cpu()
print(f"PyTorch embedding tensor shape: {tuple(embeddings_t.shape)}")

# Metadata: quantile band per sample
y_te_flat_t = y_te[:n_emb, 0]
q33, q67    = np.percentile(y_te_flat_t, [33, 67])
bands       = ['Low' if v < q33 else ('High' if v > q67 else 'Mid') for v in y_te_flat_t]
metadata    = list(zip([f'{v:.3f}' for v in y_te_flat_t], bands))

writer.add_embedding(
    mat            = embeddings_t,
    metadata       = metadata,
    metadata_header=['MedHouseVal', 'ValueBand'],
    tag            = 'hidden_3_embedding',
)
writer.flush()
print("Projector data written via add_embedding — load the PyTorch run in TensorBoard → Projector tab.")

## In-Notebook: 2D PCA of the Embedding (Projector Preview)

Reproduce what TensorBoard's Projector tab shows with PCA, directly in the notebook.

In [ ]:
from sklearn.decomposition import PCA

pca2    = PCA(n_components=2, random_state=42)
emb_2d  = pca2.fit_transform(embeddings)
y_colour = y_te_flat

plt.figure(figsize=(8, 6))
sc = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y_colour, cmap='viridis', s=10, alpha=0.7)
plt.colorbar(sc, label='MedHouseVal ($100k)')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA of hidden_3 activations — colour=MedHouseVal\n(matches TensorBoard Projector tab)')
plt.tight_layout(); plt.show()

print("If house values are smoothly ordered in this 2D space,")
print("the network has learned a useful internal representation.")

## Diagnosing Vanishing Gradients via Weight Changes

Simulate what TensorBoard's Histograms panel reveals: compare the weight mean/std at epoch 1 vs epoch 60 for each layer.

In [ ]:
# ── Record weights at start and end of training ───────────────────────────────
model_diag = build_model()

# Weights before any training
weights_before = {layer.name: layer.get_weights()[0].copy()
                  for layer in model_diag.layers if layer.get_weights()}

model_diag.fit(X_tr, y_tr, epochs=60, batch_size=256, verbose=0)

# Weights after training
weights_after = {layer.name: layer.get_weights()[0]
                 for layer in model_diag.layers if layer.get_weights()}

print(f"{'Layer':<12}  {'Before std':>12}  {'After std':>12}  {'Δ std':>12}")
for name in weights_before:
    s_b = weights_before[name].std()
    s_a = weights_after[name].std()
    print(f"{name:<12}  {s_b:>12.6f}  {s_a:>12.6f}  {s_a - s_b:>+12.6f}")

print("\nA layer where Δ std ≈ 0 across many epochs → likely vanishing gradients")

### PyTorch equivalent — weight change diagnostic

In [ ]:
torch.manual_seed(42)
diag_model = HousingNet(in_features=X_tr.shape[1]).to(device)

weights_before_t = {
    name: p.detach().clone().cpu().numpy()
    for name, p in diag_model.named_parameters() if 'weight' in name
}

opt_diag = torch.optim.Adam(diag_model.parameters(), lr=1e-3)
for _ in range(60):
    diag_model.train()
    for xb, yb in loader_tb:
        opt_diag.zero_grad()
        loss_fn(diag_model(xb), yb).backward()
        opt_diag.step()

weights_after_t = {
    name: p.detach().clone().cpu().numpy()
    for name, p in diag_model.named_parameters() if 'weight' in name
}

print(f"{'Layer':<16}  {'Before std':>12}  {'After std':>12}  {'Δ std':>12}")
for name in weights_before_t:
    s_b = weights_before_t[name].std()
    s_a = weights_after_t[name].std()
    print(f"{name:<16}  {s_b:>12.6f}  {s_a:>12.6f}  {s_a - s_b:>+12.6f}")
print("\nA layer where Δ std ≈ 0 across many epochs → likely vanishing gradients.")

## What Can Go Wrong: Logging Every Batch

Compare log directory sizes and event counts when using `update_freq='epoch'` vs `update_freq='batch'`.

In [ ]:
import shutil

for freq, label in [('epoch', 'epoch_log'), ('batch', 'batch_log')]:
    log_tmp  = os.path.join('logs', f'ch16_freq_test_{label}')
    m_tmp    = build_model()
    m_tmp.fit(
        X_tr, y_tr,
        epochs          = 5,
        batch_size      = 256,
        callbacks       = [keras.callbacks.TensorBoard(
            log_dir=log_tmp, update_freq=freq, histogram_freq=0)],
        verbose = 0
    )
    size_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(log_tmp) for f in files
    )
    n_events = 5 if freq == 'epoch' else (5 * (len(X_tr) // 256))
    print(f"update_freq='{freq:>5}': {size_bytes/1024:.1f} KB   ~{n_events} scalar entries per metric")

print("\n→ 'batch' creates ~60× more events — unusable for long runs")

### PyTorch equivalent — per-batch vs per-epoch logging cost

In PyTorch you control granularity by choosing where `add_scalar` lives: inside vs outside the
batch loop. Same lesson — per-batch logging creates hundreds of times more events.

In [ ]:
for freq, label in [('epoch', 'epoch_log_torch'), ('batch', 'batch_log_torch')]:
    log_tmp  = os.path.join('logs', f'ch16_freq_test_{label}')
    w_tmp    = SummaryWriter(log_dir=log_tmp)
    m_tmp    = HousingNet(in_features=X_tr.shape[1]).to(device)
    o_tmp    = torch.optim.Adam(m_tmp.parameters(), lr=1e-3)

    global_step = 0
    for epoch in range(5):
        m_tmp.train()
        for xb, yb in loader_tb:
            o_tmp.zero_grad()
            loss = loss_fn(m_tmp(xb), yb)
            loss.backward()
            o_tmp.step()
            if freq == 'batch':
                w_tmp.add_scalar('loss/train', loss.item(), global_step)
            global_step += 1
        if freq == 'epoch':
            w_tmp.add_scalar('loss/train', loss.item(), epoch)
    w_tmp.flush(); w_tmp.close()

    size_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(log_tmp) for f in files
    )
    n_events = 5 if freq == 'epoch' else (5 * (len(X_tr) // 256))
    print(f"update_freq='{freq:>5}': {size_bytes/1024:.1f} KB   ~{n_events} scalar entries per metric")

print("\n→ Same takeaway: per-batch logging bloats event files for no readability gain.")

## Exercises

1. **Dead neuron detection.** Modify `build_model()` to use **sigmoid** activations instead of ReLU. Train for 60 epochs and compare the hidden_1 weight std before and after training to the ReLU version. Does sigmoid show more evidence of frozen early-layer weights?

2. **Gradient norm logging.** Write a custom Keras callback that computes the L2 norm of gradients for each layer at the end of each epoch (hint: use `tf.GradientTape`) and logs them as TensorBoard scalars. Plot the gradient norms across epochs for each layer. Do the norms decrease in early layers relative to the output layer?

3. **Profiler tab.** Re-run training with `profile_batch=2` and open the TensorBoard Profile tab. Which operations dominate the per-step time? Is the bottleneck in the forward pass, backward pass, or data loading?

In [ ]:
# Exercise 1 — Dead neuron detection with sigmoid
# TODO: your solution here

In [ ]:
# Exercise 2 — Gradient norm logger
# TODO: your solution here

In [ ]:
# Exercise 3 — TensorBoard Profiler
# TODO: your solution here